# 10 · Policy Gradients
**DEEP RL** · ⏱ 50 min · 🧪 2 exercises · ⭐ 320 XP

*Optimise the policy directly — softmax policies, the log-derivative trick, the policy gradient theorem, REINFORCE and baselines.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "10-policy-gradients"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Everything so far has been **value-based**: learn q̂(s, a), then act greedily (or ε-greedily)
with respect to it. The policy was an afterthought — an argmax.

Policy gradient methods flip this around. Write down a **parameterised policy** π(a | s, θ)
and adjust θ by gradient ascent on the expected return:

```text
J(θ) = E_π[ G₀ ]          θ ← θ + α ∇J(θ)
```

No argmax, no value table needed (though values will come back as helpers).

## 01 · Why optimise the policy directly?

- **Stochastic policies.** Rock-paper-scissors, partially observed mazes, bluffing in poker:
  the best policy can be random. ε-greedy can only be "greedy plus noise"; a softmax policy can
  learn *any* probabilities.
- **Continuous actions.** argmaxₐ q(s, a) over a torque in [−2, 2] is an optimisation problem
  at every step. A policy that outputs a Gaussian over torques just *samples*.
- **Smooth changes.** A tiny change in q̂ can flip an argmax and change the behaviour
  completely. A tiny change in θ changes the action probabilities a tiny bit. This makes
  learning better behaved — and is the foundation of PPO.
- **Sometimes the policy is simpler than the values.** In CartPole "push toward the side the pole
  is falling" is a linear rule; the value function is a complicated curved surface.

## 02 · The softmax policy

For discrete actions, compute a **preference** h(s, a, θ) for every action and turn the
preferences into probabilities with a softmax:

```text
π(a | s, θ) = exp(h(s, a, θ)) / Σ_b exp(h(s, b, θ))
```

With linear preferences h(s, a) = x(s)ᵀ θₐ (one column of θ per action), this is exactly
multinomial logistic regression — the policy is a classifier whose "labels" will come from
rewards. Drag the temperature: high temperature flattens the distribution (explore), low
temperature sharpens it toward the argmax (exploit).

In [ ]:
#@title 🎮 **Interactive** · Softmax policy
rlcolab.widget("softmax", prefs=[2.0, 1.0, 0.1, -1.0], temperature=1.0)

> **💡 TIP** · **Stable softmax**
>
> exp(1000) overflows. Since softmax(h) = softmax(h − c) for any constant c, always subtract
> max(h) before exponentiating. Every deep learning library does this internally.

## 03 · The log-derivative trick

We want ∇θ E[G]. The expectation is over trajectories whose *probabilities* depend on θ, so we
cannot simply move the gradient inside. The trick: for any probability p(x; θ),

```text
∇θ p(x; θ) = p(x; θ) ∇θ log p(x; θ)
```

(just the chain rule for log). Applying it to the expected return of a one-step problem:

```text
∇θ E_{a∼π}[ r(a) ] = Σₐ r(a) ∇π(a) = Σₐ π(a) r(a) ∇log π(a) = E_{a∼π}[ r(a) ∇log π(a) ]
```

The right-hand side is an expectation under the policy — so we can **estimate it by sampling**:
take an action, observe r, and r · ∇log π(a) is an unbiased estimate of the gradient. We never
need to know how r depends on θ, and the environment need not be differentiable.

∇log π(a | s, θ) is called the **score function**. For the linear softmax it is beautifully simple:

```text
∇θ log π(a | s) = x(s) (𝟙ₐ − π(· | s))ᵀ          (one column per action)
```

Read it as: *raise the preference of the action you took, lower all actions in proportion to
how likely they already were*. Multiply by the return and good actions get pushed up, bad ones down.

### Worked example

Two actions, x(s) = [1], θ = [0, 0] so π = [0.5, 0.5]. We take a = 1 and get G = 4, α = 0.1.

- score: x · (𝟙₁ − π) = [−0.5, +0.5]
- update: θ ← [0, 0] + 0.1 · 4 · [−0.5, 0.5] = [−0.2, 0.2]
- new π = softmax([−0.2, 0.2]) ≈ [0.40, 0.60] — action 1 became more likely.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("10-policy-gradients#0", {"q": "With a softmax policy, π(·|s) = [0.7, 0.2, 0.1] and you took action 0. Which column of the score function for this state is NEGATIVE (for x(s) > 0)?", "options": ["Column 0 only", "Columns 1 and 2", "All columns", "None"], "answer": 1, "explain": "The score is x·(𝟙ₐ − π) = x·[0.3, −0.2, −0.1]. The taken action's preference goes up (by 1 − 0.7), every other action's goes down in proportion to its probability."})

---
### 🧪 Exercise 1 · A softmax policy and its score function · ⭐ **120 XP**

Build the linear-softmax policy π(a|s) = softmax(x(s)ᵀθ) — the policy you will train with REINFORCE next.

- `softmax(z)` — numerically stable (subtract the max) and working over the **last axis**, so it handles both a vector and a batch of rows.
- `policy_probs(theta, x)` — `theta` has shape `(d, n_actions)`, `x` has shape `(d,)`.
- `grad_log_pi(theta, x, a)` — the **score function** ∇_θ log π(a|s) = **outer(x, onehot(a) − π(·|s))**.
- `sample_action(theta, x, rng)` — draw a ~ π(·|s).

The check verifies your gradient against finite differences. The demo uses it to solve a 3-armed bandit by pure gradient ascent.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 10-policy-gradients/01-softmax-policy
import numpy as np
import rlhud


def softmax(z):
    """Numerically stable softmax over the last axis. Works for a vector or a batch (rows)."""
    # TODO: subtract the max along the last axis (keepdims=True), exponentiate, normalise
    raise NotImplementedError


def policy_probs(theta, x):
    """Linear-softmax policy π(·|s) = softmax(x(s)ᵀ θ).   theta: (d, n_actions), x: (d,)"""
    # TODO: one line
    raise NotImplementedError


def grad_log_pi(theta, x, a):
    """∇_θ log π(a|s) for the linear-softmax policy, same shape as theta.

    log π(a|s) = h_a − log Σ_b exp(h_b),  h = xᵀθ
    ∂ log π(a|s) / ∂θ[:, b] = x · (1[a = b] − π(b|s))      →  outer(x, onehot(a) − π)
    """
    # TODO
    raise NotImplementedError


def sample_action(theta, x, rng):
    """Draw an action from π(·|s)."""
    # TODO: rng.choice(n_actions, p=...)
    raise NotImplementedError


if __name__ == "__main__":
    # A one-state "bandit" with 3 actions and rewards 1, 2, 5 (+ noise). Gradient ascent on
    # E[r] using the score-function estimate r · ∇log π(a) — REINFORCE in its smallest form.
    rng = np.random.default_rng(0)
    theta = np.zeros((1, 3))
    x = np.array([1.0])
    means = np.array([1.0, 2.0, 5.0])
    live = rlhud.chart("π(a) while learning", xlabel="step")
    for step in range(600):
        a = sample_action(theta, x, rng)
        r = means[a] + rng.normal()
        theta += 0.02 * r * grad_log_pi(theta, x, a)
        if step % 5 == 0:
            p = policy_probs(theta, x)
            for i in range(3):
                live.add(f"π(a{i})", float(p[i]))
    live.flush()
    rlhud.bars(policy_probs(theta, x), labels=["a0 (r≈1)", "a1 (r≈2)", "a2 (r≈5)"],
               title="Final policy", highlight=2)

In [ ]:
#@title ✅ **Check** · A softmax policy and its score function (+120 XP) — tests the code you last ran above
rlcolab.check("10-policy-gradients/01-softmax-policy")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

softmax: `z = z - z.max(axis=-1, keepdims=True); e = np.exp(z); return e / e.sum(axis=-1, keepdims=True)`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

grad_log_pi: compute `p = policy_probs(theta, x)`, build a one-hot vector for `a`, subtract `p` from it.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`np.outer(x, onehot - p)` has shape (d, n_actions), like theta. sample_action: `int(rng.choice(len(p), p=p))`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("10-policy-gradients/01-softmax-policy")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 10-policy-gradients/01-softmax-policy
# Reference solution
import numpy as np
import rlhud


def softmax(z):
    """Numerically stable softmax over the last axis. Works for a vector or a batch (rows)."""
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=-1, keepdims=True)    # subtracting a constant does not change softmax
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def policy_probs(theta, x):
    """Linear-softmax policy π(·|s) = softmax(x(s)ᵀ θ).   theta: (d, n_actions), x: (d,)"""
    return softmax(x @ theta)


def grad_log_pi(theta, x, a):
    """∇_θ log π(a|s) for the linear-softmax policy, same shape as theta.

    log π(a|s) = h_a − log Σ_b exp(h_b),  h = xᵀθ
    ∂ log π(a|s) / ∂θ[:, b] = x · (1[a = b] − π(b|s))      →  outer(x, onehot(a) − π)
    """
    p = policy_probs(theta, x)
    onehot = np.zeros_like(p)
    onehot[a] = 1.0
    return np.outer(x, onehot - p)


def sample_action(theta, x, rng):
    """Draw an action from π(·|s)."""
    p = policy_probs(theta, x)
    return int(rng.choice(len(p), p=p))


if __name__ == "__main__":
    # A one-state "bandit" with 3 actions and rewards 1, 2, 5 (+ noise). Gradient ascent on
    # E[r] using the score-function estimate r · ∇log π(a) — REINFORCE in its smallest form.
    rng = np.random.default_rng(0)
    theta = np.zeros((1, 3))
    x = np.array([1.0])
    means = np.array([1.0, 2.0, 5.0])
    live = rlhud.chart("π(a) while learning", xlabel="step")
    for step in range(600):
        a = sample_action(theta, x, rng)
        r = means[a] + rng.normal()
        theta += 0.02 * r * grad_log_pi(theta, x, a)
        if step % 5 == 0:
            p = policy_probs(theta, x)
            for i in range(3):
                live.add(f"π(a{i})", float(p[i]))
    live.flush()
    rlhud.bars(policy_probs(theta, x), labels=["a0 (r≈1)", "a1 (r≈2)", "a2 (r≈5)"],
               title="Final policy", highlight=2)

## 04 · The policy gradient theorem

For full episodes the same trick gives the **policy gradient theorem**: the gradient of J only
involves the score of each action, weighted by how good things went *afterwards*:

```text
∇J(θ) ∝ E_π[ Σₜ Gₜ ∇θ log π(Aₜ | Sₜ, θ) ]
```

Remarkably the gradient does **not** involve the derivative of the state distribution — the
part that depends on the unknown environment dynamics. That is what makes model-free policy
gradients possible.

Note the use of **Gₜ**, the return *from step t on* ("reward-to-go"), not G₀: an action can't
affect rewards that came before it, so including them only adds noise.

## 05 · REINFORCE

Monte Carlo policy gradient (Williams, 1992):

```python
loop:
    play an episode with π_θ: S₀, A₀, R₁, …, S_{T−1}, A_{T−1}, R_T
    for t = 0 … T−1:
        Gₜ = Σₖ γᵏ⁻ᵗ⁻¹ Rₖ          # return-to-go
        θ ← θ + α Gₜ ∇log π(Aₜ | Sₜ, θ)
```

It is unbiased and dead simple. Its problem is **variance**. In CartPole every reward is +1,
so every Gₜ is positive: every action taken gets pushed up, and the only learning signal is
that actions in long episodes get pushed up *more*. The gradient estimate swings wildly from
episode to episode.

## 06 · Baselines: subtract what you expected

Replace Gₜ by Gₜ − b(Sₜ) for any baseline b that does not depend on the action:

```text
∇J(θ) ∝ E_π[ Σₜ (Gₜ − b(Sₜ)) ∇log π(Aₜ | Sₜ, θ) ]
```

This adds **no bias**, because the score has zero mean under the policy:

```text
E_{a∼π}[ b(s) ∇log π(a|s) ] = b(s) Σₐ ∇π(a|s) = b(s) ∇ Σₐ π(a|s) = b(s) ∇1 = 0
```

but it can reduce variance enormously. The natural choice is b(s) = v̂(s, w), a learned value
function — then Gₜ − v̂(Sₜ) is the **advantage**: was this outcome better or worse than usual
from here? Better-than-expected actions are reinforced, worse-than-expected ones discouraged,
even when all rewards are positive.

Two more variance-reduction tricks used everywhere in practice:

- **Batch several episodes** per update and average the gradient.
- **Normalise the advantages** within a batch to mean 0, std 1. This acts like an adaptive
  baseline and step size, and makes the learning rate far less sensitive to reward scale.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("10-policy-gradients#1", {"q": "Why does subtracting a baseline b(s) not bias the policy gradient?", "options": ["Because b(s) is usually small", "Because E[∇log π(a|s)] = 0 under the policy, so the b(s) term averages out", "Because the baseline is learned by gradient descent too", "It does bias it, but only a little"], "answer": 1, "explain": "Σₐ π ∇log π = Σₐ ∇π = ∇Σₐ π = ∇1 = 0. Any action-independent baseline multiplies a zero-mean quantity, so it changes the variance but not the expectation."})

---
### 🧪 Exercise 2 · REINFORCE with a baseline on CartPole · ⭐ **200 XP**

Train a **linear-softmax policy** on CartPole with Monte Carlo policy gradient.

Implement:
- `returns_to_go(rewards, gamma)` — Gₜ for every step, backwards.
- `reinforce_update(theta, w, X, A, G, lr, lr_v, normalize=True)` — one update from a batch of time steps:
  1. Âₜ = Gₜ − wᵀxₜ (learned linear baseline)
  2. w ← w + lr_v · mean(Â x)  — regress the baseline toward the returns
  3. optionally normalise Â to mean 0, std 1
  4. **θ ← θ + lr · mean(Â · outer(x, onehot(a) − π))**

Update `theta` and `w` in place and return them. The training loop (provided) plays 5 episodes per update. The check trains 1000 episodes (≈ 5 s) and requires a mean return ≥ 150. The demo also runs plain REINFORCE (no baseline) on the same budget for comparison.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 10-policy-gradients/02-reinforce-baseline
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: the policy (exercise 1)
def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def features(obs):
    """x(s) = [x, ẋ, θ, θ̇, 1] — the raw CartPole observation plus a bias term."""
    return np.append(obs, 1.0)


def sample_action(theta, x, rng):
    p = softmax(x @ theta)
    return min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)


# ---------------------------------------------------------------- your code
def returns_to_go(rewards, gamma):
    """[G0, G1, …, G_{T−1}] with Gₜ = Rₜ₊₁ + γ Gₜ₊₁ (computed backwards)."""
    # TODO (you wrote this in lesson 1!)
    raise NotImplementedError


def reinforce_update(theta, w, X, A, G, lr, lr_v, normalize=True):
    """One REINFORCE-with-baseline update from a batch of N time steps. Updates theta and w
    in place and returns them.

    X : (N, d) features of every visited state      A : (N,) actions taken
    G : (N,) return-to-go from each step            w : (d,) linear baseline b(s) = wᵀx(s)

    1. advantages   Âₜ = Gₜ − b(Sₜ)
    2. baseline     w ← w + lr_v · mean_t[ Âₜ x(Sₜ) ]                (regress b toward G)
    3. if normalize: Â ← (Â − mean) / (std + 1e-8)
    4. policy       θ ← θ + lr · mean_t[ Âₜ ∇log π(Aₜ|Sₜ) ],  ∇log π = outer(x, onehot(a) − π)
    """
    # TODO: follow the four steps in the docstring. Vectorise over the batch:
    #   softmax(X @ theta) gives π for every row; np.eye(n_actions)[A] gives the one-hots;
    #   Σ_t Âₜ outer(xₜ, onehotₜ − πₜ) = X.T @ ((onehot − P) * adv[:, None])
    raise NotImplementedError


# ---------------------------------------------------------------- provided: training loop
def train(episodes=1000, seed=0, lr=2.0, lr_v=0.1, gamma=0.99, batch_episodes=5, normalize=True, log=None):
    """REINFORCE with a learned linear baseline on CartPole. Returns (theta, w, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    theta = np.zeros((5, env.n_actions))
    w = np.zeros(5)
    returns, X, A, G = [], [], [], []
    for ep in range(episodes):
        obs, _ = env.reset()
        rewards, done = [], False
        while not done:                                  # 1) play a whole episode with π_θ
            x = features(obs)
            a = sample_action(theta, x, rng)
            obs, r, terminated, truncated, _ = env.step(a)
            X.append(x); A.append(a); rewards.append(r)
            done = terminated or truncated
        G.append(returns_to_go(rewards, gamma))          # 2) returns from every step
        returns.append(sum(rewards))
        if log:
            log(ep, returns[-1])
        if (ep + 1) % batch_episodes == 0:               # 3) update from a few episodes at once
            reinforce_update(theta, w, np.array(X), np.array(A), np.concatenate(G), lr, lr_v, normalize)
            X, A, G = [], [], []
    return theta, w, returns


def evaluate(theta, episodes=10, seed=1000):
    """Mean return of the STOCHASTIC policy (we sample, as REINFORCE does)."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            obs, r, term, trunc, _ = env.step(sample_action(theta, features(obs), rng))
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("REINFORCE on CartPole — episode return", smooth=25, xlabel="episode")
    theta, w, returns = train(episodes=1000, seed=0, log=lambda ep, G: live.add("with baseline", G))
    # Same budget without baseline or normalisation, for comparison:
    def raw(ep, G):
        live.add("no baseline, no normalisation", G)
    train(episodes=1000, seed=0, lr=0.02, lr_v=0.0, normalize=False, log=raw)
    live.flush()
    rlhud.stat("Evaluation (10 episodes)", evaluate(theta))
    rlhud.bars(theta[:, 1] - theta[:, 0], labels=["x", "ẋ", "θ", "θ̇", "bias"],
               title="Preference for RIGHT over LEFT, per feature")

In [ ]:
#@title ✅ **Check** · REINFORCE with a baseline on CartPole (+200 XP) — tests the code you last ran above
rlcolab.check("10-policy-gradients/02-reinforce-baseline")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Compute `adv = G - X @ w` once, at the start, and use it for the baseline update; normalise a copy for the policy step.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Batch probabilities: `P = softmax(X @ theta)` has shape (N, 2). One-hots: `np.eye(theta.shape[1])[A]`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`theta += lr * X.T @ ((onehot - P) * adv[:, None]) / N` — that is the mean of Âₜ·outer(xₜ, onehotₜ − πₜ).

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("10-policy-gradients/02-reinforce-baseline")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 10-policy-gradients/02-reinforce-baseline
# Reference solution
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: the policy (exercise 1)
def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def features(obs):
    """x(s) = [x, ẋ, θ, θ̇, 1] — the raw CartPole observation plus a bias term."""
    return np.append(obs, 1.0)


def sample_action(theta, x, rng):
    p = softmax(x @ theta)
    return min(int(np.searchsorted(np.cumsum(p), rng.random(), side="right")), len(p) - 1)


# ---------------------------------------------------------------- your code
def returns_to_go(rewards, gamma):
    """[G0, G1, …, G_{T−1}] with Gₜ = Rₜ₊₁ + γ Gₜ₊₁ (computed backwards)."""
    G, out = 0.0, np.zeros(len(rewards))
    for t in reversed(range(len(rewards))):
        G = rewards[t] + gamma * G
        out[t] = G
    return out


def reinforce_update(theta, w, X, A, G, lr, lr_v, normalize=True):
    """One REINFORCE-with-baseline update from a batch of N time steps. Updates theta and w
    in place and returns them.

    X : (N, d) features of every visited state      A : (N,) actions taken
    G : (N,) return-to-go from each step            w : (d,) linear baseline b(s) = wᵀx(s)

    1. advantages   Âₜ = Gₜ − b(Sₜ)
    2. baseline     w ← w + lr_v · mean_t[ Âₜ x(Sₜ) ]                (regress b toward G)
    3. if normalize: Â ← (Â − mean) / (std + 1e-8)
    4. policy       θ ← θ + lr · mean_t[ Âₜ ∇log π(Aₜ|Sₜ) ],  ∇log π = outer(x, onehot(a) − π)
    """
    N = len(A)
    adv = G - X @ w
    w += lr_v * (X.T @ adv) / N
    if normalize:
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    P = softmax(X @ theta)                              # (N, n_actions)
    onehot = np.eye(theta.shape[1])[A]
    theta += lr * X.T @ ((onehot - P) * adv[:, None]) / N
    return theta, w


# ---------------------------------------------------------------- provided: training loop
def train(episodes=1000, seed=0, lr=2.0, lr_v=0.1, gamma=0.99, batch_episodes=5, normalize=True, log=None):
    """REINFORCE with a learned linear baseline on CartPole. Returns (theta, w, episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    theta = np.zeros((5, env.n_actions))
    w = np.zeros(5)
    returns, X, A, G = [], [], [], []
    for ep in range(episodes):
        obs, _ = env.reset()
        rewards, done = [], False
        while not done:                                  # 1) play a whole episode with π_θ
            x = features(obs)
            a = sample_action(theta, x, rng)
            obs, r, terminated, truncated, _ = env.step(a)
            X.append(x); A.append(a); rewards.append(r)
            done = terminated or truncated
        G.append(returns_to_go(rewards, gamma))          # 2) returns from every step
        returns.append(sum(rewards))
        if log:
            log(ep, returns[-1])
        if (ep + 1) % batch_episodes == 0:               # 3) update from a few episodes at once
            reinforce_update(theta, w, np.array(X), np.array(A), np.concatenate(G), lr, lr_v, normalize)
            X, A, G = [], [], []
    return theta, w, returns


def evaluate(theta, episodes=10, seed=1000):
    """Mean return of the STOCHASTIC policy (we sample, as REINFORCE does)."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            obs, r, term, trunc, _ = env.step(sample_action(theta, features(obs), rng))
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("REINFORCE on CartPole — episode return", smooth=25, xlabel="episode")
    theta, w, returns = train(episodes=1000, seed=0, log=lambda ep, G: live.add("with baseline", G))
    # Same budget without baseline or normalisation, for comparison:
    def raw(ep, G):
        live.add("no baseline, no normalisation", G)
    train(episodes=1000, seed=0, lr=0.02, lr_v=0.0, normalize=False, log=raw)
    live.flush()
    rlhud.stat("Evaluation (10 episodes)", evaluate(theta))
    rlhud.bars(theta[:, 1] - theta[:, 0], labels=["x", "ẋ", "θ", "θ̇", "bias"],
               title="Preference for RIGHT over LEFT, per feature")

> **🔑 KEY IDEA** · **Push up actions that did better than expected**
>
> Policy gradient = Σ (advantage) × ∇log π(action). The score ∇log π says *how* to make an action
> more likely; the advantage says *whether* to. Everything that follows — actor-critic, PPO —
> is a better way to estimate the advantage or a safer way to take the step.

## 07 · Debugging policy gradients

- **Check the sign.** We *ascend* J. If returns drop steadily from the start, you are descending.
  In a loss-based framework the loss is −mean(Â · log π).
- **Watch the entropy** H(π) = −Σ π log π. If it collapses to 0 early, the policy became
  deterministic before it found anything good — lower the learning rate or add an entropy bonus.
- **Watch the probabilities.** Print π(·|s) for a few fixed states during training. Slowly
  moving probabilities are healthy; flipping to 0/1 in a handful of updates means steps are too big.
- **Learning rate sensitivity is real.** Without normalisation the gradient scales with the
  return (here up to ~100), so the right α can differ by orders of magnitude between tasks.
- **One seed is an anecdote.** Policy gradient curves are noisy; compare methods over several seeds.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("10-policy-gradients#2", {"q": "REINFORCE's entropy drops to almost zero within the first 20 updates and the return is stuck at ~20. What should you try first?", "options": ["Increase the learning rate", "Decrease the learning rate and/or add an entropy bonus", "Remove the baseline", "Increase γ to 1"], "answer": 1, "explain": "The policy collapsed to a deterministic, bad behaviour and can no longer explore. Smaller steps or an entropy bonus keep it stochastic long enough to find better actions."})

## 08 · If you have PyTorch

With autograd you never write the score function — you write a "loss" whose gradient *is* the
policy gradient:

```python
logits = policy_net(torch.as_tensor(X))                 # (N, n_actions)
logp = torch.log_softmax(logits, -1).gather(1, A[:, None]).squeeze(1)
adv = (G - baseline).detach()                           # no gradient through the advantage
adv = (adv - adv.mean()) / (adv.std() + 1e-8)
loss = -(adv * logp).mean()                             # minus: optimisers minimise
opt.zero_grad(); loss.backward(); opt.step()
```

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("10-policy-gradients")